# Project 2: data sanity checks for the five shortlisted general datasets

**Purpose.** Before any modelling, check each shortlisted dataset for problems in the data itself. No model is trained in this notebook.

| # | Dataset | Kaggle |
|---|---|---|
| 1 | Credit Risk (loan default) | `laotse/credit-risk-dataset` |
| 2 | Default of Credit Card Clients (Taiwan) | `uciml/default-of-credit-card-clients-dataset` |
| 3 | Online Shoppers Purchasing Intention | `imakash3011/online-shoppers-purchasing-intention-dataset` |
| 4 | Bank Customer Churn | `gauravtopre/bank-customer-churn-dataset` |
| 5 | Bank Marketing (term deposits) | `henriqueyamahata/bank-marketing` |

**Two kinds of check**
1. **Generic, the same for every dataset:** size and types; missing values (counting `?`, `unknown` and blanks); duplicate rows and duplicates with conflicting labels; constant and ID-like columns; class balance; rare category levels; extreme values; strongly correlated pairs; and a leakage scan.
2. **Specific to each dataset:** rules a correct record must satisfy, written from what each column means (for example, years employed can't exceed age minus 14).

The leakage scan is the only check that uses the label. It runs on an 80% development split, so the 20% test split is held out from the start. Everything else uses features only.

## 0. Setup and shared checks

In [1]:
import os, re, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
SEED = 42
MISSING_TOKENS = {"?", "unknown", "Unknown", "NA", "N/A", ""}
SUMMARY = {}

def load(slug, pattern=r"\.csv$"):
    root = kagglehub.dataset_download(slug)
    files = [os.path.join(dp, fn) for dp, _, fns in os.walk(root) for fn in fns
             if fn.lower().endswith(".csv") and re.search(pattern, fn, re.I)]
    f = max(files, key=os.path.getsize)
    with open(f, errors="ignore") as fh:
        head = fh.readline()
    df = pd.read_csv(f, sep=";" if head.count(";") > head.count(",") else ",", low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"loaded {os.path.basename(f)}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

def say(msg):
    display(Markdown(msg))

def generic_checks(name, df, target, y, id_cols=(), leak_cols=()):
    """The same checks for every dataset. Prints tables and stores a one-line summary."""
    X = df.drop(columns=[target])
    num = X.select_dtypes(include="number").columns.tolist()
    txt = [c for c in X.columns if c not in num]
    s = {}

    say("**Size and types**")
    print(f"rows {len(df):,}; feature columns {X.shape[1]} ({len(num)} numeric, {len(txt)} text); memory {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    say("**Missing values** (NaN, plus `?`, `unknown`, `NA` and blanks in text columns)")
    nan = X.isna().sum()
    tok = pd.Series({c: int(X[c].astype(str).str.strip().isin(MISSING_TOKENS).sum()) if c in txt else 0 for c in X.columns})
    m = pd.DataFrame({"NaN": nan, "token": tok}); m["share"] = (m.NaN + m.token) / len(X)
    m = m[m.share > 0].sort_values("share", ascending=False)
    display(m.style.format({"share": "{:.2%}"}) if len(m) else pd.DataFrame({"result": ["no missing values"]}))
    s["missing cells"] = f"{(nan.sum() + tok.sum()) / X.size:.2%}"

    say("**Duplicates** (on the columns a model would use: IDs and known post-outcome columns left out)")
    use = X.drop(columns=[c for c in list(id_cols) + list(leak_cols) if c in X.columns])
    dup = int(pd.concat([use, y], axis=1).duplicated().sum())
    h = pd.util.hash_pandas_object(use, index=False)
    g = pd.DataFrame({"h": h.values, "y": y.values}).groupby("h")["y"].agg(["nunique", "size"])
    conflict_groups = int((g["nunique"] > 1).sum()); conflict_rows = int(g.loc[g["nunique"] > 1, "size"].sum())
    print(f"duplicate rows (features and label identical): {dup:,} ({dup / len(df):.2%})")
    print(f"identical features with different labels: {conflict_groups:,} groups, {conflict_rows:,} rows")
    s["duplicate rows"] = f"{dup:,} ({dup / len(df):.1%})"; s["conflicting-label rows"] = f"{conflict_rows:,}"

    say("**Constant, near-constant and ID-like columns**")
    nun = X.nunique(dropna=False)
    top = X.apply(lambda c: c.value_counts(dropna=False, normalize=True).iloc[0])
    idlike = [c for c in X.columns if c in id_cols or (nun[c] == len(X))]
    rows = [(c, "constant") for c in X.columns if nun[c] <= 1] + [(c, f"near-constant ({top[c]:.1%} one value)") for c in X.columns if nun[c] > 1 and top[c] >= 0.99] + [(c, "ID-like (unique per row)") for c in idlike]
    display(pd.DataFrame(rows, columns=["column", "issue"]) if rows else pd.DataFrame({"result": ["none"]}))
    s["constant / ID columns"] = ", ".join(r[0] for r in rows) or "none"

    say("**Class balance**")
    pos = int(y.sum())
    print(f"positives {pos:,} of {len(y):,} ({y.mean():.1%}); expected in a 20% validation split: about {round(pos * 0.2):,}")
    s["positive rate"] = f"{y.mean():.1%}"; s["validation positives"] = f"{round(pos * 0.2):,}"

    if txt:
        say("**Category levels** (rare = under 1% of rows)")
        lv = []
        for c in txt:
            vc = X[c].value_counts(normalize=True)
            lv.append({"column": c, "levels": len(vc), "rare levels": int((vc < 0.01).sum()), "rarest level": f"{vc.index[-1]} ({vc.iloc[-1]:.2%}, {int(vc.iloc[-1] * len(X))} rows)"})
        display(pd.DataFrame(lv))

    say("**Ranges and spikes** (numeric columns; `max ÷ 99th pct` above 3 means a few values sit far beyond the rest)")
    ex = []
    for c in num:
        if c in idlike: continue
        v = X[c].dropna().astype(float); p1, p99 = v.quantile([0.01, 0.99])
        ex.append({"column": c, "min": v.min(), "1st pct": p1, "median": v.median(), "99th pct": p99, "max": v.max(),
                   "max ÷ 99th pct": v.max() / p99 if p99 > 0 else np.nan, "share of zeros": (v == 0).mean()})
    ex = pd.DataFrame(ex).sort_values("max ÷ 99th pct", ascending=False)
    display(ex.style.format({c: "{:,.4g}" for c in ["min", "1st pct", "median", "99th pct", "max", "max ÷ 99th pct"]} | {"share of zeros": "{:.1%}"}))
    s["columns with spikes (max > 3 × 99th pct)"] = ", ".join(ex.loc[ex["max ÷ 99th pct"] > 3, "column"]) or "none"

    say("**Strongly correlated numeric pairs** (|r| ≥ 0.9)")
    cn = [c for c in num if c not in idlike and X[c].nunique() > 1]
    r = X[cn].corr().abs()
    pairs = [(a, b, round(r.loc[a, b], 3)) for i, a in enumerate(cn) for b in cn[i + 1:] if r.loc[a, b] >= 0.9]
    display(pd.DataFrame(pairs, columns=["a", "b", "|r|"]) if pairs else pd.DataFrame({"result": ["none"]}))
    s["correlated pairs"] = str(len(pairs))

    say("**Leakage scan** (single-feature AUC on the 80% development split; 0.5 = no signal, ≥ 0.95 = suspicious)")
    dev, _ = train_test_split(np.arange(len(X)), test_size=0.2, stratify=y, random_state=SEED)
    Xd, yd = X.iloc[dev], y.iloc[dev]
    au = {}
    for c in X.columns:
        if c in idlike: continue
        v = Xd[c]
        if c in txt:                                   # a text column: score each level by its positive rate
            rate = yd.groupby(v.astype(str)).mean(); v = v.astype(str).map(rate)
        v = pd.to_numeric(v, errors="coerce"); v = v.fillna(v.median())
        if v.nunique() > 1:
            a = roc_auc_score(yd, v); au[c] = max(a, 1 - a)
    au = pd.Series(au).sort_values(ascending=False)
    print("strongest single features:", ", ".join(f"{c} {v:.3f}" for c, v in au.head(5).items()))
    s["strongest single feature"] = f"{au.index[0]} ({au.iloc[0]:.3f})"
    for c in leak_cols:
        if c in au: print(f"known post-outcome column `{c}`: AUC {au[c]:.3f} on its own")
    SUMMARY[name] = s

def rule_table(rules):
    """rules: list of (rule, boolean mask of violating rows, what to do). Shows counts and shares."""
    n = len(rules[0][1])
    t = pd.DataFrame([{"rule a correct record should satisfy": r, "violations": int(m.sum()), "share": m.mean(), "decision": d} for r, m, d in rules])
    display(t.style.format({"share": "{:.2%}"}))
    return t
print("ready")

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ready


---
## 1. Credit Risk (`laotse/credit-risk-dataset`)
One row per loan applicant: age, income, home ownership, years employed, loan purpose, grade, amount, interest rate, loan as a share of income, past default on file, credit history length. Target `loan_status` (1 = defaulted).

In [2]:
cr = load("laotse/credit-risk-dataset")
y_cr = cr["loan_status"].astype(int)
generic_checks("Credit Risk", cr, "loan_status", y_cr)

loaded credit_risk_dataset.csv: 32,581 rows x 12 columns


**Size and types**

rows 32,581; feature columns 11 (7 numeric, 4 text); memory 9.0 MB


**Missing values** (NaN, plus `?`, `unknown`, `NA` and blanks in text columns)

,NaN,token,share
loan_int_rate,3116,0,9.56%
person_emp_length,895,0,2.75%


**Duplicates** (on the columns a model would use: IDs and known post-outcome columns left out)

... (9 outputs left out)

**Strongly correlated numeric pairs** (|r| ≥ 0.9)

,result
0,none


**Leakage scan** (single-feature AUC on the 80% development split; 0.5 = no signal, ≥ 0.95 = suspicious)

strongest single features: loan_grade 0.721, loan_percent_income 0.721, loan_int_rate 0.705, person_income 0.691, person_home_ownership 0.652


In [3]:
say("**Validity rules**")
calc = cr["loan_amnt"] / cr["person_income"]
t = rule_table([
    ("age is at most 100", cr["person_age"] > 100, "Drop: no applicant is over 100 (max is 144)."),
    ("years employed ≤ age − 14", cr["person_emp_length"] > cr["person_age"] - 14, "Drop: impossible."),
    ("credit history ≤ age − 16 years", cr["cb_person_cred_hist_length"] > cr["person_age"] - 16, "Check: possible data-entry error."),
    ("loan_percent_income ≈ loan amount / income (within 0.01)", (cr["loan_percent_income"] - calc).abs() > 0.01, "Keep; it is a derived column, so it adds no new information."),
    ("interest rate is recorded", cr["loan_int_rate"].isna(), "Impute (median by loan grade) or drop."),
    ("years employed is recorded", cr["person_emp_length"].isna(), "Impute or drop."),
])
print("loan grade vs default rate (higher grade letters should default more):")
display(cr.groupby("loan_grade")["loan_status"].agg(["size", "mean"]).round(3).T)

**Validity rules**

,rule a correct record should satisfy,violations,share,decision
0,age is at most 100,5,0.02%,Drop: no applicant is over 100 (max is 144).
1,years employed ≤ age − 14,2,0.01%,Drop: impossible.
2,credit history ≤ age − 16 years,0,0.00%,Check: possible data-entry error.
3,loan_percent_income ≈ loan amount / income (within 0.01),388,1.19%,"Keep; it is a derived column, so it adds no new information."
4,interest rate is recorded,3116,9.56%,Impute (median by loan grade) or drop.
5,years employed is recorded,895,2.75%,Impute or drop.


loan grade vs default rate (higher grade letters should default more):


loan_grade,A,B,C,D,E,F,G
size,10777.0,10451.000,6458.000,3626.00,964.000,241.000,64.000
mean,0.1,0.163,0.207,0.59,0.644,0.705,0.984


---
## 2. Default of Credit Card Clients (`uciml/default-of-credit-card-clients-dataset`)
Taiwan, 2005. One row per card holder: credit limit, sex, education, marital status, age, repayment status for the last six months (`PAY_0`, `PAY_2`…`PAY_6`), bill amounts and payment amounts for the same months. Target `default.payment.next.month`.

The dataset's documentation says: education 1 = graduate school, 2 = university, 3 = high school, 4 = others; marital status 1 = married, 2 = single, 3 = others; repayment status −1 = paid duly, 1–9 = months of delay.

In [4]:
cc = load("uciml/default-of-credit-card-clients-dataset")
y_cc = cc["default.payment.next.month"].astype(int)
generic_checks("Credit Card Default", cc, "default.payment.next.month", y_cc, id_cols=["ID"])

loaded UCI_Credit_Card.csv: 30,000 rows x 25 columns


**Size and types**

rows 30,000; feature columns 24 (24 numeric, 0 text); memory 6.0 MB


**Missing values** (NaN, plus `?`, `unknown`, `NA` and blanks in text columns)

,result
0,no missing values


**Duplicates** (on the columns a model would use: IDs and known post-outcome columns left out)

... (7 outputs left out)

**Strongly correlated numeric pairs** (|r| ≥ 0.9)

,a,b,|r|
0,BILL_AMT1,BILL_AMT2,0.951
1,BILL_AMT2,BILL_AMT3,0.928
2,BILL_AMT3,BILL_AMT4,0.924
3,BILL_AMT4,BILL_AMT5,0.940
4,BILL_AMT4,BILL_AMT6,0.901
5,BILL_AMT5,BILL_AMT6,0.946


**Leakage scan** (single-feature AUC on the 80% development split; 0.5 = no signal, ≥ 0.95 = suspicious)

strongest single features: PAY_0 0.693, PAY_2 0.640, PAY_3 0.627, LIMIT_BAL 0.623, PAY_4 0.613


In [5]:
say("**Validity rules**")
pay = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill = [f"BILL_AMT{i}" for i in range(1, 7)]
t = rule_table([
    ("education is a documented code (1–4)", ~cc["EDUCATION"].isin([1, 2, 3, 4]), "Merge 0, 5 and 6 into 4 (others)."),
    ("marital status is a documented code (1–3)", ~cc["MARRIAGE"].isin([1, 2, 3]), "Merge 0 into 3 (others)."),
    ("every repayment status is a documented code (−1, 1–9)", ~cc[pay].isin([-1] + list(range(1, 10))).all(axis=1), "Codes −2 and 0 are undocumented; treat repayment status as categories, not numbers."),
    ("no bill amount is negative", (cc[bill] < 0).any(axis=1), "Keep: a negative bill is a credit balance (overpayment)."),
    ("latest bill ≤ 1.5 × credit limit", cc["BILL_AMT1"] > 1.5 * cc["LIMIT_BAL"], "Keep, but note: the balance is far over the limit."),
])
print("column names skip PAY_1 (PAY_0 then PAY_2): rename to PAY_1…PAY_6 for clarity.")
print("repayment-status codes in PAY_0:", cc["PAY_0"].value_counts().sort_index().to_dict())
print("default rate by PAY_0:"); display(cc.groupby("PAY_0")["default.payment.next.month"].mean().round(3).to_frame().T)

**Validity rules**

,rule a correct record should satisfy,violations,share,decision
0,education is a documented code (1–4),345,1.15%,"Merge 0, 5 and 6 into 4 (others)."
1,marital status is a documented code (1–3),54,0.18%,Merge 0 into 3 (others).
2,"every repayment status is a documented code (−1, 1–9)",25939,86.46%,"Codes −2 and 0 are undocumented; treat repayment status as categories, not numbers."
3,no bill amount is negative,1930,6.43%,Keep: a negative bill is a credit balance (overpayment).
4,latest bill ≤ 1.5 × credit limit,164,0.55%,"Keep, but note: the balance is far over the limit."


column names skip PAY_1 (PAY_0 then PAY_2): rename to PAY_1…PAY_6 for clarity.
repayment-status codes in PAY_0: {-2: 2759, -1: 5686, 0: 14737, 1: 3688, 2: 2667, 3: 322, 4: 76, 5: 26, 6: 11, 7: 9, 8: 19}
default rate by PAY_0:


PAY_0,-2,-1,0,1,2,3,4,5,6,7,8
default.payment.next.month,0.132,0.168,0.128,0.339,0.691,0.758,0.684,0.5,0.545,0.778,0.579


---
## 3. Online Shoppers Purchasing Intention (`imakash3011/online-shoppers-purchasing-intention-dataset`)
One row per website session over a year: number of administrative, informational and product pages visited and time spent on each, Google Analytics bounce rate, exit rate and page value, closeness to a special day, month, operating system, browser, region, traffic source, visitor type, weekend. Target `Revenue` (the session ended in a purchase).

In [6]:
os_ = load("imakash3011/online-shoppers-purchasing-intention-dataset")
y_os = os_["Revenue"].astype(int)
generic_checks("Online Shoppers", os_, "Revenue", y_os)

loaded online_shoppers_intention.csv: 12,330 rows x 18 columns


**Size and types**

rows 12,330; feature columns 17 (14 numeric, 3 text); memory 2.8 MB


**Missing values** (NaN, plus `?`, `unknown`, `NA` and blanks in text columns)

,result
0,no missing values


**Duplicates** (on the columns a model would use: IDs and known post-outcome columns left out)

... (9 outputs left out)

**Strongly correlated numeric pairs** (|r| ≥ 0.9)

,a,b,|r|
0,BounceRates,ExitRates,0.913


**Leakage scan** (single-feature AUC on the 80% development split; 0.5 = no signal, ≥ 0.95 = suspicious)

strongest single features: PageValues 0.867, ExitRates 0.711, ProductRelated_Duration 0.675, ProductRelated 0.660, Administrative 0.631


In [7]:
say("**Validity rules**")
rules = []
for p in ["Administrative", "Informational", "ProductRelated"]:
    rules.append((f"{p}: time spent > 0 only if pages > 0", (os_[p] == 0) & (os_[f"{p}_Duration"] > 0), "Check: time recorded without a page."))
rules += [
    ("a session visits at least one page", os_[["Administrative", "Informational", "ProductRelated"]].sum(axis=1) == 0, "Check: empty sessions."),
    ("bounce rate ≤ exit rate", os_["BounceRates"] > os_["ExitRates"], "Keep: a bounce is a kind of exit, so this should hold; the share shows how noisy the analytics are."),
    ("bounce and exit rates within 0–0.2", (os_[["BounceRates", "ExitRates"]] > 0.2).any(axis=1) | (os_[["BounceRates", "ExitRates"]] < 0).any(axis=1), "Keep."),
]
t = rule_table(rules)
print("months present:", sorted(os_["Month"].unique()), "→ January and April are missing: the year is incomplete.")
print("OperatingSystems, Browser, Region and TrafficType are integer codes with no published labels: treat them as categories.")
print("PageValues is 0 for", f"{(os_['PageValues'] == 0).mean():.1%}", "of sessions; purchase rate when 0:", f"{y_os[os_['PageValues'] == 0].mean():.1%}", "and when > 0:", f"{y_os[os_['PageValues'] > 0].mean():.1%}")

**Validity rules**

,rule a correct record should satisfy,violations,share,decision
0,Administrative: time spent > 0 only if pages > 0,0,0.00%,Check: time recorded without a page.
1,Informational: time spent > 0 only if pages > 0,0,0.00%,Check: time recorded without a page.
2,ProductRelated: time spent > 0 only if pages > 0,0,0.00%,Check: time recorded without a page.
3,a session visits at least one page,6,0.05%,Check: empty sessions.
4,bounce rate ≤ exit rate,175,1.42%,"Keep: a bounce is a kind of exit, so this should hold; the share shows how noisy the analytics are."
5,bounce and exit rates within 0–0.2,0,0.00%,Keep.


months present: ['Aug', 'Dec', 'Feb', 'Jul', 'June', 'Mar', 'May', 'Nov', 'Oct', 'Sep'] → January and April are missing: the year is incomplete.
OperatingSystems, Browser, Region and TrafficType are integer codes with no published labels: treat them as categories.
PageValues is 0 for 77.9% of sessions; purchase rate when 0: 3.9% and when > 0: 56.3%


---
## 4. Bank Customer Churn (`gauravtopre/bank-customer-churn-dataset`)
One row per bank customer: credit score, country, gender, age, tenure, balance, number of products, has a credit card, is an active member, estimated salary. Target `churn`.

In [8]:
bc = load("gauravtopre/bank-customer-churn-dataset")
y_bc = bc["churn"].astype(int)
generic_checks("Bank Customer Churn", bc, "churn", y_bc, id_cols=["customer_id"])

loaded Bank Customer Churn Prediction.csv: 10,000 rows x 12 columns


**Size and types**

rows 10,000; feature columns 11 (9 numeric, 2 text); memory 1.9 MB


**Missing values** (NaN, plus `?`, `unknown`, `NA` and blanks in text columns)

,result
0,no missing values


**Duplicates** (on the columns a model would use: IDs and known post-outcome columns left out)

... (9 outputs left out)

**Strongly correlated numeric pairs** (|r| ≥ 0.9)

,result
0,none


**Leakage scan** (single-feature AUC on the 80% development split; 0.5 = no signal, ≥ 0.95 = suspicious)

strongest single features: age 0.728, country 0.601, active_member 0.594, products_number 0.584, balance 0.579


In [9]:
say("**Validity rules**")
t = rule_table([
    ("credit score within 350–850", ~bc["credit_score"].between(350, 850), "Keep."),
    ("credit score not stuck at a bound", bc["credit_score"].isin([350, 850]), "Keep, but note: values piled at the bounds look capped."),
    ("age at least 18", bc["age"] < 18, "Drop."),
    ("tenure ≤ age", bc["tenure"] > bc["age"], "Drop: impossible."),
    ("estimated salary at least 1,000", bc["estimated_salary"] < 1000, "Check: implausibly low salaries."),
    ("balance recorded as exactly 0", bc["balance"] == 0, "Keep, but note: a large spike at 0."),
])
print("churn rate by number of products:"); display(bc.groupby("products_number")["churn"].agg(["size", "mean"]).round(3).T)
print("estimated_salary looks uniform (no link to age or country), which suggests generated data:")
display(bc["estimated_salary"].describe().round(0).to_frame().T)

**Validity rules**

,rule a correct record should satisfy,violations,share,decision
0,credit score within 350–850,0,0.00%,Keep.
1,credit score not stuck at a bound,238,2.38%,"Keep, but note: values piled at the bounds look capped."
2,age at least 18,0,0.00%,Drop.
3,tenure ≤ age,0,0.00%,Drop: impossible.
4,"estimated salary at least 1,000",59,0.59%,Check: implausibly low salaries.
5,balance recorded as exactly 0,3617,36.17%,"Keep, but note: a large spike at 0."


churn rate by number of products:


products_number,1,2,3,4
size,5084.000,4590.000,266.000,60.0
mean,0.277,0.076,0.827,1.0


estimated_salary looks uniform (no link to age or country), which suggests generated data:


,count,mean,std,min,25%,50%,75%,max
estimated_salary,10000.0,100090.0,57510.0,12.0,51002.0,100194.0,149388.0,199992.0


---
## 5. Bank Marketing (`henriqueyamahata/bank-marketing`)
Portugal, May 2008 to November 2010. One row per phone call in a term-deposit campaign: client details, contact details, campaign history, and monthly economic indicators. Target `y` (the client subscribed).

The dataset's own documentation says `duration` (call length) is known only after the call, and should be dropped for a realistic model. `pdays = 999` means the client was not contacted before.

In [10]:
bm = load("henriqueyamahata/bank-marketing", r"bank-additional-full")
y_bm = (bm["y"] == "yes").astype(int)
generic_checks("Bank Marketing", bm, "y", y_bm, leak_cols=["duration"])

loaded bank-additional-full.csv: 41,188 rows x 21 columns


**Size and types**

rows 41,188; feature columns 20 (10 numeric, 10 text); memory 28.1 MB


**Missing values** (NaN, plus `?`, `unknown`, `NA` and blanks in text columns)

,NaN,token,share
default,0,8597,20.87%
education,0,1731,4.20%
housing,0,990,2.40%
loan,0,990,2.40%
job,0,330,0.80%
marital,0,80,0.19%


**Duplicates** (on the columns a model would use: IDs and known post-outcome columns left out)

... (9 outputs left out)

**Strongly correlated numeric pairs** (|r| ≥ 0.9)

,a,b,|r|
0,emp.var.rate,euribor3m,0.972
1,emp.var.rate,nr.employed,0.907
2,euribor3m,nr.employed,0.945


**Leakage scan** (single-feature AUC on the 80% development split; 0.5 = no signal, ≥ 0.95 = suspicious)

strongest single features: duration 0.817, nr.employed 0.745, euribor3m 0.740, emp.var.rate 0.713, month 0.659
known post-outcome column `duration`: AUC 0.817 on its own


In [11]:
say("**Validity rules**")
t = rule_table([
    ("pdays = 999 (not contacted before) exactly when previous = 0", (bm["pdays"] == 999) != (bm["previous"] == 0), "Check: the two columns should agree."),
    ("poutcome = 'nonexistent' exactly when previous = 0", (bm["poutcome"] == "nonexistent") != (bm["previous"] == 0), "Check."),
    ("a call with duration 0 did not end in a subscription", (bm["duration"] == 0) & (bm["y"] == "yes"), "Confirms duration is decided after the outcome."),
    ("default is not 'yes'", bm["default"] == "yes", "Only 3 rows: merge or drop the level."),
    ("default is recorded", bm["default"] == "unknown", "Keep 'unknown' as its own level (it is informative)."),
    ("campaign (calls this campaign) ≤ 20", bm["campaign"] > 20, "Keep, but cap for a linear model."),
])
print("pdays = 999 is a placeholder, not a number of days: use a 'contacted before' flag instead.")
print("the five economic indicators take only", bm[["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]].nunique().to_dict(), "values: they are monthly figures, so they encode time.")
print("rows are in time order: subscription rate in the first and last 10% of rows:", f"{y_bm.iloc[:len(bm)//10].mean():.1%}", "vs", f"{y_bm.iloc[-len(bm)//10:].mean():.1%}")

**Validity rules**

,rule a correct record should satisfy,violations,share,decision
0,pdays = 999 (not contacted before) exactly when previous = 0,4110,9.98%,Check: the two columns should agree.
1,poutcome = 'nonexistent' exactly when previous = 0,0,0.00%,Check.
2,a call with duration 0 did not end in a subscription,0,0.00%,Confirms duration is decided after the outcome.
3,default is not 'yes',3,0.01%,Only 3 rows: merge or drop the level.
4,default is recorded,8597,20.87%,Keep 'unknown' as its own level (it is informative).
5,campaign (calls this campaign) ≤ 20,157,0.38%,"Keep, but cap for a linear model."


pdays = 999 is a placeholder, not a number of days: use a 'contacted before' flag instead.
the five economic indicators take only {'emp.var.rate': 10, 'cons.price.idx': 26, 'cons.conf.idx': 26, 'euribor3m': 316, 'nr.employed': 11} values: they are monthly figures, so they encode time.
rows are in time order: subscription rate in the first and last 10% of rows: 2.8% vs 45.9%


---
## 6. Summary of the generic checks

In [12]:
display(pd.DataFrame(SUMMARY))

,Credit Risk,Credit Card Default,Online Shoppers,Bank Customer Churn,Bank Marketing
missing cells,1.12%,0.00%,0.00%,0.00%,1.54%
duplicate rows,165 (0.5%),35 (0.1%),125 (1.0%),0 (0.0%),"1,784 (4.3%)"
conflicting-label rows,0,46,0,0,542
constant / ID columns,none,ID,none,customer_id,none
positive rate,21.8%,22.1%,15.5%,20.4%,11.3%
validation positives,"1,422","1,327",382,407,928
columns with spikes (max > 3 × 99th pct),"person_income, person_emp_length","PAY_AMT2, PAY_AMT1, PAY_AMT3, PAY_AMT4, PAY_AMT5, PAY_AMT6, BILL_AMT3, BILL_...","ProductRelated_Duration, PageValues, Administrative_Duration, Informational,...",none,"campaign, duration, previous"
correlated pairs,0,6,1,0,3
strongest single feature,loan_grade (0.721),PAY_0 (0.693),PageValues (0.867),age (0.728),duration (0.817)


## 7. Findings: what each dataset needs before modelling
All numbers are from the run above. "Cleaning" means a change to the data; "decision" means a choice to state on the slides.

| Dataset | Verdict | Cleaning needed | Decisions to state |
|---|---|---|---|
| **Bank Customer Churn** | Cleanest | None: no missing values, no duplicates, no impossible values | The data looks generated: estimated salary is spread evenly from £12 to £200,000 with no link to anything; 36% of balances are exactly 0; 238 credit scores sit exactly at 350 or 850. Customers with 3–4 products churn 83–100% of the time (326 customers), a near-certain pocket. Churn is also the course's own example. |
| **Default of Credit Card Clients** | Clean, real data | Drop 46 rows in 21 groups with identical features but different labels, and 35 duplicate rows. Merge undocumented codes: education 0, 5, 6 (345 rows) into "others"; marital status 0 (54 rows) into "others" | Repayment-status codes −2 and 0 are undocumented and appear in 86% of rows, so treat repayment status as categories, not numbers. Bill and payment amounts are heavy-tailed (the largest payment is 22 × the 99th percentile): log-transform them for the linear model. Six bill-amount pairs correlate at 0.90–0.95. `PAY_1` is missing from the names (`PAY_0`, then `PAY_2`). |
| **Credit Risk** | Mostly clean | Drop 7 impossible rows (5 applicants older than 100, up to 144; 2 employed longer than age − 14) and 165 duplicate rows. Fill `loan_int_rate` (**9.6% missing**) and `person_emp_length` (2.7%) | `loan_grade` and `loan_int_rate` are the lender's own risk ratings. If the grade is set using information close to the outcome, it could act as a near-leak; grade G defaults 98% of the time (64 loans). Income has one extreme value (6,000,000, 27 × the 99th percentile). `loan_percent_income` is just amount ÷ income. |
| **Online Shoppers** | Clean, but one feature is a leakage risk | Drop 125 duplicate rows (1%) and 6 sessions with no pages | **`PageValues` may leak the label.** Google Analytics computes page value from transactions that followed the page, so it can contain the purchase it is meant to predict: purchase rate is 56% when it is above 0 and 3.9% when it is 0. Either remove it or argue why it is known before the purchase. Also: January and April are missing, four columns are unlabelled integer codes (treat as categories), most duration columns are 48–80% zeros. |
| **Bank Marketing** | Most work | Drop `duration` (known only after the call; AUC 0.817 alone). Then drop 1,784 duplicate rows (4.3%) and 542 rows in 237 groups with conflicting labels. Recode `pdays = 999` (a placeholder, not days) as a flag; merge `default = yes` (3 rows) | **The rows are in time order and the outcome drifts strongly**: 2.8% subscribe in the first tenth of the rows and 45.9% in the last. The five economic indicators take only 10–26 values each and encode the month. A random split would mix past and future, so this dataset needs a time-ordered split. "unknown" makes up 20.9% of `default`, 4.2% of `education`. 4,110 rows have `pdays = 999` but `previous > 0`. |

### Ranking on data quality alone
1. **Default of Credit Card Clients**: real data; the fixes are small and easy to explain (drop 81 rows, merge undocumented codes).
2. **Credit Risk**: one column with 9.6% missing values and 7 impossible rows; otherwise clean. Ask whether `loan_grade` is fair to use.
3. **Bank Customer Churn**: no cleaning at all, but it looks generated and is the course's own example.
4. **Online Shoppers**: clean, but the strongest feature may contain the answer.
5. **Bank Marketing**: the most cleaning, and it needs a time-ordered split.

Merged with the modelling screening in `Project2_General_Dataset_Screening.ipynb`, **Default of Credit Card Clients** and **Credit Risk** are the two strongest candidates. Credit Card Default is real data with the lightest cleaning; Credit Risk has the clearer model gap.